In [1]:
import numpy as np
# import torch
# import gpytorch
# from gpytorch.models import ApproximateGP
# from gpytorch.variational import CholeskyVariationalDistribution, VariationalStrategy
# from gpytorch.means import ConstantMean
# from gpytorch.kernels import RBFKernel, ScaleKernel
# from gpytorch.likelihoods import GaussianLikelihood
# from gpytorch.distributions import MultivariateNormal
from sklearn.cluster import KMeans

In [2]:
# Générer des données synthétiques non-stationnaires
np.random.seed(0)
n = 1000
X = np.linspace(0, 10, n).reshape(-1, 1)
Y = np.sin(X).ravel() + 0.1 * np.random.randn(n)
Y[n//2:] += 0.5 * X[n//2:].ravel()  # Changement de régime

# Clusteriser les données (pour simuler les experts)
kmeans = KMeans(n_clusters=2).fit(X)
clusters = kmeans.labels_


In [3]:

# Définir un modèle GP pour chaque cluster
class GPModel(ApproximateGP):
    def __init__(self, inducing_points):
        variational_distribution = CholeskyVariationalDistribution(inducing_points.size(0))
        variational_strategy = VariationalStrategy(
            self, inducing_points, variational_distribution, learn_inducing_locations=True
        )
        super().__init__(variational_strategy)
        self.mean_module = ConstantMean()
        self.covar_module = ScaleKernel(RBFKernel())
        self.likelihood = GaussianLikelihood()

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return MultivariateNormal(mean_x, covar_x)

# Entraîner un GP par cluster
models = []
likelihoods = []
for k in range(2):
    X_k = torch.tensor(X[clusters == k], dtype=torch.float32)
    Y_k = torch.tensor(Y[clusters == k], dtype=torch.float32)
    inducing_points = X_k[::10, :]  # Points d'induction
    model = GPModel(inducing_points)
    likelihood = GaussianLikelihood()
    models.append(model)
    likelihoods.append(likelihood)

# Boucle d'entraînement (simplifiée)
for model, likelihood, X_k, Y_k in zip(models, likelihoods, [X[clusters == k] for k in range(2)], [Y[clusters == k] for k in range(2)]):
    model.train()
    likelihood.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
    for _ in range(100):
        optimizer.zero_grad()
        output = model(X_k)
        loss = -likelihood.log_marginal(output, Y_k)
        loss.backward()
        optimizer.step()

# Prédiction par mélange
def predict(X_test):
    X_test = torch.tensor(X_test, dtype=torch.float32)
    with torch.no_grad():
        preds = []
        for model, likelihood in zip(models, likelihoods):
            pred = likelihood(model(X_test))
            preds.append(pred.mean.numpy())
        # Moyenne pondérée (ici, poids égaux pour simplicité)
        return np.mean(preds, axis=0)

# Test
X_test = np.linspace(0, 10, 100).reshape(-1, 1)
Y_pred = predict(X_test)


NameError: name 'ApproximateGP' is not defined